# Complete Mamba Block

> Build the full Mamba selective state space model block

- **Level:** Intermediate
- **Time:** ~25 minutes
- **GPU:** Recommended

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Parallel scan function
def ssm_parallel_scan(A_bar, B_bar):
    """Parallel scan for SSM"""
    batch, seq_len, d_state = B_bar.shape
    n = 1 << (seq_len - 1).bit_length()
    
    A = torch.zeros(batch, n, d_state, device=A_bar.device, dtype=A_bar.dtype)
    B = torch.zeros(batch, n, d_state, device=B_bar.device, dtype=B_bar.dtype)
    A[:, :seq_len, :] = A_bar
    B[:, :seq_len, :] = B_bar
    
    for stride in range(n.bit_length() - 1):
        step = 1 << stride
        A_even = A[:, ::step*2, :]
        B_even = B[:, ::step*2, :]
        A_odd = A[:, step::step*2, :]
        B_odd = B[:, step::step*2, :]
        
        A[:, step::step*2, :] = A_even * A_odd
        B[:, step::step*2, :] = A_even * B_odd + B_even
    
    return B[:, :seq_len, :]
# export
ssm_parallel_scan = ssm_parallel_scan

## Selective SSM

In [ ]:
class SelectiveSSM(nn.Module):
    """Mamba's Selective State Space Model"""
    
    def __init__(self, d_model, d_state, dt_rank=16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.dt_rank = dt_rank
        
        # A: fixed state dynamics
        self.A_log = nn.Parameter(torch.randn(d_state))
        self.D = nn.Parameter(torch.ones(d_state))
        
        # Selective projections
        self.x_proj = nn.Linear(d_model, dt_rank + d_state * 2, bias=False)
        self.out_proj = nn.Linear(d_state, d_model, bias=False)
    
    def forward(self, x):
        batch, seq_len, d_model = x.shape
        d_state = self.d_state
        
        # Compute selective parameters
        x_params = self.x_proj(x)
        delta = x_params[:, :, :self.dt_rank]
        B = x_params[:, :, self.dt_rank:self.dt_rank + d_state]
        C = x_params[:, :, self.dt_rank + d_state:]
        
        # Softplus for positive delta
        delta = F.softplus(delta)
        
        # Discretize
        A_bar = torch.exp(self.A_log.unsqueeze(0).unsqueeze(0) * delta.unsqueeze(-1))
        
        disc = (A_bar - 1) / torch.where(
            torch.abs(self.A_log.unsqueeze(0).unsqueeze(0)) > 1e-6,
            self.A_log.unsqueeze(0).unsqueeze(0),
            1e-6
        )
        B_bar = disc * B.unsqueeze(-1)
        
        # Parallel scan
        h = ssm_parallel_scan(A_bar, B_bar)
        
        # Output
        y = torch.einsum('bldn,bln->bld', h, C)
        y = y + x * self.D.unsqueeze(0).unsqueeze(0)
        
        return self.out_proj(y)
# export
SelectiveSSM = SelectiveSSM

## Full Mamba Block

In [ ]:
class MambaBlock(nn.Module):
    """Complete Mamba Block with gating and normalization"""
    
    def __init__(self, d_model, d_state=128, expand=2, dt_rank=16):
        super().__init__()
        self.d_model = d_model
        self.d_inner = int(expand * d_model)
        
        # Pre-norm
        self.norm = nn.LayerNorm(d_model)
        
        # Input projection with expansion
        self.in_proj = nn.Linear(d_model, self.d_inner * 2)
        
        # Selective SSM
        self.ssm = SelectiveSSM(self.d_inner, d_state, dt_rank)
        
        # Output projection
        self.out_proj = nn.Linear(self.d_inner, d_model)
    
    def forward(self, x):
        # Pre-norm
        x_norm = self.norm(x)
        
        # Input projection and split for gating
        x_inner, gate = self.in_proj(x_norm).chunk(2, dim=-1)
        
        # SiLU gating
        gate = F.silu(gate)
        
        # SSM
        ssm_out = self.ssm(gate)
        
        # Apply gate
        y = ssm_out * gate
        
        # Output projection and residual
        return self.out_proj(y) + x
# export
MambaBlock = MambaBlock

In [ ]:
# Test Mamba Block
block = MambaBlock(d_model=256, d_state=128).to(device)
x = torch.randn(2, 32, 256).to(device)
y = block(x)
print(f"Input: {x.shape} → Output: {y.shape}")
print(f"Parameters: {sum(p.numel() for p in block.parameters()):,}")

# Verify gradients
loss = y.sum()
loss.backward()
print("✅ Backpropagation works!")

## Benchmark vs Transformer

In [ ]:
class TransformerBlock(nn.Module):
    """Standard Transformer for comparison"""
    def __init__(self, d_model, n_heads=8):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model)
        )
    
    def forward(self, x):
        x = x + self.attn(self.norm(x), self.norm(x), self.norm(x))[0]
        x = x + self.ff(self.norm(x))
        return x

# Benchmark
def benchmark(model, seq_len, n_runs=10):
    x = torch.randn(4, seq_len, 256).to(device)
    with torch.no_grad():
        _ = model(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs):
            y = model(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    return (time.perf_counter() - start) / n_runs * 1000

seq_lens = [64, 128, 256, 512, 1024]
print("Seq Len | Mamba (ms) | Transformer (ms) | Speedup")
print("-" * 55)

for seq_len in seq_lens:
    mamba = MambaBlock(d_model=256, d_state=128).to(device)
    transformer = TransformerBlock(d_model=256).to(device)
    
    t_mamba = benchmark(mamba, seq_len)
    t_trans = benchmark(transformer, seq_len)
    
    print(f"{seq_len:7} | {t_mamba:9.2f} | {t_trans:16.2f} | {t_trans/t_mamba:.2f}x")

## Summary

What we built:

| Component | Description |
|----------|-------------|
| SelectiveSSM | Input-dependent B, C, Δ |
| MambaBlock | Full block with gating, norm |
| Benchmark | Shows O(N) vs O(N²) scaling |

Key insights:

1. **Selection** allows content-based reasoning
2. **SiLU gating** enables adaptive computation
3. **Pre-norm** stabilizes training
4. **Mamba scales better** with sequence length

---

**Next:** Train a language model in the next notebook!